# L03 — запуск очистки
Основной код находится в `prepare_data.py`. Этот ноутбук запускает его без копирования логики. Готовый результат полного запуска уже лежит в `processed/`. Все числа в отчёте рассчитываются из CSV.

В Colab загрузите полный ZIP через панель файлов. На компьютере распакуйте ZIP и откройте этот ноутбук из папки `alpha_l03`.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    archive = Path("/content/AlphaHack_L03_complete.zip")
    if not Path("/content/alpha_l03/prepare_data.py").exists():
        with zipfile.ZipFile(archive) as z:
            z.extractall("/content")
    ROOT = Path("/content/alpha_l03")
else:
    ROOT = Path.cwd()
assert (ROOT / "prepare_data.py").is_file(), "Укажите ROOT — папку с prepare_data.py"
print(ROOT)


## Установка зависимости
Выполняется в вашей среде. При первом запуске нужен доступ к установке пакетов.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")], check=True)


## Пути и полный запуск
Для повторного запуска выберите новое имя OUT. Скрипт не перезаписывает непустые папки. На всех трёх CSV обработка может занять несколько минут.

In [ ]:
OUT = ROOT / ("processed_colab" if IN_COLAB else "processed_new")
command = [sys.executable, str(ROOT / "prepare_data.py"), "--persons", str(ROOT / "raw/person_links.csv"), "--companies", str(ROOT / "raw/companies.csv"), "--payments", str(ROOT / "raw/payments.csv"), "--out", str(OUT)]
subprocess.run(command, check=True)


## Отчёт и просмотр первых записей

In [ ]:
from IPython.display import display, Markdown
import json
import pandas as pd
display(Markdown((OUT / "cleaning_report.md").read_text(encoding="utf-8")))
for name in ["person_links_prepared", "companies_prepared", "payments_prepared"]:
    print(name)
    display(pd.read_csv(OUT / f"{name}.csv.gz", dtype="string", nrows=5, keep_default_na=False))


## Проверка кода

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "-v"], cwd=ROOT, check=True)


## Скачать новый результат из Colab (необязательно)
Исходники не включаются в этот архив. Передавайте результаты в закрытом хранилище команды. Перед загрузкой ноутбука в GitHub очистите его выводы.

In [ ]:
import shutil
result_zip = shutil.make_archive(str(ROOT / "L03_processed_result"), "zip", OUT)
if IN_COLAB:
    from google.colab import files
    files.download(result_zip)
else:
    print(result_zip)
